# CIC-UNSW-NB15 - Network-Flow Analysis & Graph Visualisation

This notebook explores the **CIC-UNSW-NB15** intrusion-detection dataset that we loaded into Neo4j in [`loader.ipynb`](loader.ipynb). It has two halves:

1. **Dataset statistics** - `seaborn` charts driven by aggregate Cypher queries against the live graph.
2. **Graph visualisations** - sub-graphs pulled from Neo4j, laid out with [`neo4j-viz`](https://github.com/neo4j/neo4j-viz), rendered to PNG via **Playwright** (using the helper in [`neo4j_analysis.py`](neo4j_analysis.py)), saved under `renderings/`, and inlined below.

The graph is a **star / fact model**: every flow is reified as a `:Flow` node wired to shared dimension nodes (`:Host`, `:Port`, `:Protocol`, `:AttackCategory`, `:Subnet`). The full load holds **3.54 M flows** between **43 hosts**.


## Setup

Connect to Neo4j through the `Neo4jAnalysis` helper and set a consistent plotting theme.

In [ ]:
import os
import asyncio
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from IPython.display import Image, display
from neo4j_analysis import Neo4jAnalysis

warnings.filterwarnings("ignore")
load_dotenv()

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titleweight"] = "bold"

# A stable colour for "malicious vs benign" used throughout
BENIGN_C, ATTACK_C = "#4c9f70", "#d1495b"

analysis = Neo4jAnalysis(
    os.getenv("NEO4J_URI"),
    os.getenv("NEO4J_USERNAME"),
    os.getenv("NEO4J_PASSWORD"),
    os.getenv("NEO4J_DATABASE"),
)
print("connection ok:", analysis.verify_connection())

RENDER_DIR = os.getenv("RENDER_DIR", "renderings")
os.makedirs(RENDER_DIR, exist_ok=True)

## The graph schema

The schema loaded into Neo4j is a star / fact model, with `:Flow` nodes at the centre of the graph, connected to shared dimension nodes representing hosts, ports, protocols, attack categories, and subnets. This structure allows for efficient querying and analysis of the relationships between different entities in the dataset.

In [ ]:
from neo4j_viz.neo4j import from_neo4j


async def render_and_capture(VG, name, width=1600, height=1000, scale=2):
    """Render a VisualizationGraph to renderings/<name>.png via the Playwright helper."""
    html = VG.render(width=f"{width}px", height=f"{height}px")
    html_path = os.path.join(RENDER_DIR, f"{name}.html")
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html.data)
    png_path = os.path.join(RENDER_DIR, f"{name}.png")
    # Passing html_file= reuses the saved layout and skips the centering-CSS injection,
    # which otherwise resizes the NVL canvas and clips the graph.
    await analysis.capture_graph_to_png(
        None, png_path, width=width, height=height, scale=scale, html_file=html_path
    )
    print("saved", png_path)
    return png_path

In [ ]:
schema = analysis.run_query_viz(
    """CALL db.schema.visualization() YIELD nodes, relationships
    RETURN nodes, relationships"""
)
VG = from_neo4j(schema)
png = await render_and_capture(VG, "schema", width=1600, height=1000, scale=1.5)
display(Image(filename=png))

## Dataset statistics


### How (im)balanced is the data?

Intrusion datasets are dominated by normal traffic - the interesting attack classes are a tiny minority. We pull the per-class flow counts straight from the graph.


In [ ]:
cat = analysis.run_query_df("""
    MATCH (f:Flow)-[:CLASSIFIED_AS]->(c:AttackCategory)
    RETURN c.name AS category, c.malicious AS malicious, count(f) AS flows
    ORDER BY flows DESC
    """)
total = cat["flows"].sum()
cat["pct"] = 100 * cat["flows"] / total
display(cat)

fig, ax = plt.subplots(figsize=(12, 6))
palette = [ATTACK_C if m else BENIGN_C for m in cat["malicious"]]
sns.barplot(data=cat, x="flows", y="category", palette=palette, ax=ax)
ax.set_xscale("log")
ax.set_xlabel("flows (log scale)")
ax.set_ylabel("")
ax.set_title("Flows per attack category  -  benign dwarfs every attack class")
for p, (_, row) in zip(ax.patches, cat.iterrows()):
    ax.text(
        p.get_width() * 1.08,
        p.get_y() + p.get_height() / 2,
        f"{int(row.flows):,} ({row.pct:.2f}%)",
        va="center",
        fontsize=11,
    )
ax.set_xlim(right=ax.get_xlim()[1] * 6)
plt.tight_layout()
plt.show()

print(
    f"Malicious flows: {cat.loc[cat.malicious, 'flows'].sum():,} "
    f"({100 * cat.loc[cat.malicious, 'flows'].sum() / total:.2f}% of all traffic)"
)

### Composition of the *attack* traffic

Drop the overwhelming benign class and look only at how the malicious flows split across the nine attack categories.


In [ ]:
atk = cat[cat["malicious"]].copy()

fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(
    data=atk,
    x="category",
    y="flows",
    palette=sns.color_palette("flare", len(atk)),
    ax=ax,
)
ax.set_title("Attack-only flow counts  -  Exploits & Fuzzers dominate")
ax.set_xlabel("")
ax.set_ylabel("flows")
plt.xticks(rotation=35, ha="right")
for p in ax.patches:
    ax.text(
        p.get_x() + p.get_width() / 2,
        p.get_height() + atk["flows"].max() * 0.01,
        f"{int(p.get_height()):,}",
        ha="center",
        va="bottom",
        fontsize=10,
    )
plt.tight_layout()
plt.show()

### Which services are attackers aiming at?

Join `:Flow` to its `:Port` dimension and keep only malicious flows. The `service` annotation (well-known-port → name) was attached during the load.


In [ ]:
svc = analysis.run_query_df("""
    MATCH (c:AttackCategory {malicious: true})<-[:CLASSIFIED_AS]-(f:Flow)-[:ON_PORT]->(p:Port)
RETURN p.number AS port,
       coalesce(p.service, 'port ' + toString(p.number)) AS service,
       count(f) AS attackFlows
ORDER BY attackFlows DESC LIMIT 12
    """)
svc["label"] = svc["service"] + "  (" + svc["port"].astype(str) + ")"
display(svc[["port", "service", "attackFlows"]])

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(
    data=svc,
    x="attackFlows",
    y="label",
    palette=sns.color_palette("rocket_r", len(svc)),
    ax=ax,
)
ax.set_title("Top 12 targeted services (malicious flows only)")
ax.set_xlabel("attack flows")
ax.set_ylabel("")
for p in ax.patches:
    ax.text(
        p.get_width() * 1.01,
        p.get_y() + p.get_height() / 2,
        f"{int(p.get_width()):,}",
        va="center",
        fontsize=10,
    )
plt.tight_layout()
plt.show()

### Where does the attack traffic flow? (subnet → subnet)

The `:Subnet` dimension cleanly separates the **attacker range** from the **victim range**. Aggregating the `:CONNECTED_TO` edges by subnet gives an attack-traffic matrix.


In [ ]:
edges = analysis.run_query_df("""
    MATCH (:Host)-[:IN_SUBNET]->(sa:Subnet)
    WITH sa
    MATCH (sa)<-[:IN_SUBNET]-(:Host)-[c:CONNECTED_TO]->(:Host)-[:IN_SUBNET]->(da:Subnet)
    RETURN sa.cidr AS src_subnet, da.cidr AS dst_subnet,
           sum(c.attackFlows) AS attackFlows
    ORDER BY attackFlows DESC
    """)
mat = edges.pivot_table(
    index="src_subnet", columns="dst_subnet", values="attackFlows", fill_value=0
)

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(
    mat,
    annot=True,
    fmt=",.0f",
    cmap="rocket_r",
    linewidths=0.5,
    cbar_kws={"label": "attack flows"},
    ax=ax,
)
ax.set_title("Attack flows: source subnet  →  destination subnet")
ax.set_xlabel("destination subnet")
ax.set_ylabel("source subnet")
plt.tight_layout()
plt.show()


## Graph visualisations


### All flows categorised as 'worm' activity

Let us start by identifying the flows that are categorised as **worm** activity. We can query the graph for all flows that have an `:AttackCategory` of "Worm" and visualise the resulting subgraph, including the source and destination hosts, ports, and protocols involved in these flows. This will help us understand the spread and impact of worm attacks within the network.

In [ ]:
g = analysis.run_query_viz(
    """
    MATCH (src:Host)-[s:SENDS]->(f:Flow)-[op:ON_PORT]->(p:Port),
          (f)-[cl:CLASSIFIED_AS]->(cat:AttackCategory {name: 'Worms'}),
          (f)-[t:TO]->(dst:Host)
    RETURN src, f, p, cat, dst, s, op, cl, t
    LIMIT 100
    """
)
VG = from_neo4j(g)

CAPTION_PROP = {
    "Host": "ip",
    "Port": "service",
    "AttackCategory": "name",
    "Flow": "label",
}

for node in VG.nodes:
    label = node.properties.get("labels", ["?"])[0]
    node.properties["group"] = label
    cap = node.properties.get(CAPTION_PROP.get(label, ""))
    if label == "Port" and not cap:
        cap = str(node.properties.get("number", ""))
    node.caption = str(cap) if cap is not None else label

VG.color_nodes(property="group")
png = await render_and_capture(VG, "worms_star_graph", width=1600, height=1100)
display(Image(filename=png))

print(
    f"Worms star graph: {len(VG.nodes)} nodes, {len(VG.relationships)} edges rendered."
)

### The attack graph: who hits whom

Filter to the `:CONNECTED_TO` edges that actually carry attack traffic (`attackFlows > 0`). Hosts are coloured by **role** - attackers vs victims - and sized by the number of attack flows. The structure is stark: a handful of attacker hosts fanning into the victim subnet.


In [ ]:
g = analysis.run_query_viz(
    "MATCH p=(:Host)-[c:CONNECTED_TO]->(:Host) WHERE c.attackFlows > 0 RETURN p"
)
VG = from_neo4j(g)

atk_by_ip = {}
for r in analysis.run_query(
    """MATCH (h:Host)-[c:CONNECTED_TO]-() WHERE c.attackFlows > 0
       RETURN h.ip AS ip, sum(c.attackFlows) AS af"""
):
    atk_by_ip[r["ip"]] = r["af"]

sizes = {}
for node in VG.nodes:
    ip = node.properties.get("ip", "")
    node.properties["role"] = "attacker" if ip.startswith("175.45.176.") else "victim"
    node.caption = ip
    sizes[node.id] = float(atk_by_ip.get(ip, 0)) + 1.0

VG.color_nodes(property="role", colors=[ATTACK_C, BENIGN_C])
VG.resize_nodes(sizes=sizes, node_radius_min_max=(15, 30))

png = await render_and_capture(VG, "g2_attack_graph")
display(Image(filename=png))

### An attacker's-eye view (fact-model star)

Zoom into the single "busiest" attacker and expand a sample of its malicious flows through the full star schema: `(:Host)-[:SENDS]->(:Flow)` and each flow out to its `:Port`, `:AttackCategory`, and victim `:Host`. Nodes are coloured by their **label**, so you can read the model directly: one attacker, many flow-facts, fanning into a few services and attack classes.


In [ ]:
# Determine who the top attacker is
top_attacker = analysis.run_query_df("""
    MATCH (a:Host)-[s:SENDS]->(f:Flow)
    WHERE f.label <> 'Benign'
    RETURN a.ip AS ip, COUNT(f) AS attack_count
    ORDER BY attack_count DESC
    LIMIT 1
    """).iloc[0]["ip"]

print(f"Top attacker IP: {top_attacker}")

In [ ]:
g = analysis.run_query_viz(
    """
    MATCH (a:Host {ip:$ip})-[s:SENDS]->(f:Flow)-[op:ON_PORT]->(p:Port),
          (f)-[cl:CLASSIFIED_AS]->(cat:AttackCategory),
          (f)-[t:TO]->(v:Host)
    WHERE f.label <> 'Benign'
    RETURN a, f, p, cat, v, s, op, cl, t LIMIT 120
    """,
    {"ip": top_attacker},
)
VG = from_neo4j(g)

CAPTION_PROP = {
    "Host": "ip",
    "Port": "service",
    "AttackCategory": "name",
    "Flow": "label",
}
for node in VG.nodes:
    label = node.properties.get("labels", ["?"])[0]
    node.properties["group"] = label
    cap = node.properties.get(CAPTION_PROP.get(label, ""))
    if label == "Port" and not cap:
        cap = str(node.properties.get("number", ""))
    node.caption = str(cap) if cap is not None else label

VG.color_nodes(property="group")
png = await render_and_capture(VG, "g3_attacker_star", width=1600, height=1100)
display(Image(filename=png))
print(
    f"Star graph for attacker {top_attacker}: {len(VG.nodes)} nodes, {len(VG.relationships)} edges"
)

### Wrap-up

**Statistical takeaways**
- The data is extremely imbalanced - **97.5 % benign**, with `Exploits` and `Fuzzers` the
  largest attack classes.
- Attacks concentrate on a few services (notably HTTP/port 80) and flow **exclusively**
  from `175.45.176.0/24` into `149.171.126.0/24`.

**Graph takeaways**
- The host topology is two dense internal cliques plus the attacker range.
- The attack graph reduces to a clean attacker→victim fan-out.
- The fact-model star makes a single attacker's behaviour legible in one hop.


In [ ]:
analysis.close()
print("Neo4j driver closed.")